In [0]:
from pyspark.sql import DataFrame

def create_autoloader(landing_path: str, checkpoint_path: str) -> DataFrame:
    return (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .load(landing_path)
    )


def write_autloader_on_table(bronze_table_name: str, checkpoint_path: str, df_incremental: DataFrame) -> None:
    query = (df_incremental.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(bronze_table_name))
    query.awaitTermination()